# 2026Q3差区间的内部走势与持仓机制
不按未来收益挑日子或改状态定义；保留全部64日、全部三个月和两条成本路径。仅做描述，不构建交易规则。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/q3_failure_mechanisms_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
def load(name,artifact):
 spec=plan['sources'][name];p=root/spec['card'];s=json.loads((p/'summary.json').read_text());mp=p/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==spec['manifest_sha256'];entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/'+artifact))
 with R.uri_context(s['recorder_uri']):
  rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_name=p.name);path=Path(rec.download_artifact(artifact));assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];return rec.load_object(artifact)
prices=load('constituents','analysis/prices')['$adj_close'].unstack('instrument');weights=load('constituents','analysis/weights');raw_cash=load('cash','analysis/daily').set_index('signal_date').frozen_current_cash_weight
ret1=prices.pct_change(fill_method=None);ret5=prices/prices.shift(5)-1.;low20=prices.rolling(20,min_periods=20).min();descriptors=[]
for day,part in weights.groupby('datetime'):
 w=part.set_index('instrument').weight_pct;members=w.index;w=w/w.sum();r=ret1.loc[day,members];r5=ret5.loc[day,members];newlow=prices.loc[day,members].le(low20.loc[day,members]);assert np.isfinite(r).all() and np.isfinite(r5).all() and low20.loc[day,members].notna().all()
 downside=w*(-r).clip(lower=0);loss_total=downside.sum()
 descriptors.append({'signal_date':day,'positive_weight5':float(w.loc[r5.gt(0)].sum()),'new_low_weight20':float(w.loc[newlow].sum()),'downside_top5_share':float(downside.nlargest(5).sum()/loss_total) if loss_total>0 else np.nan,'declining_members':int(r.lt(0).sum()),'largest_downside_names':','.join(downside.loc[downside.gt(0)].nlargest(5).index),'filing_date':part.filing_date.iloc[0],'report_date':part.report_date.iloc[0]})
descriptors=pd.DataFrame(descriptors).set_index('signal_date');assert len(descriptors)==64
components=['reference_cash','additional_cash','soxl_relative','recovery_gross','execution_cost','benchmark_residual','native_reconciliation_residual','relative_log'];monthly=[];groups=[];all_daily={}
for cost in plan['costs']:
 account=load('attribution','analysis/daily/current_anchor_'+str(cost));part=account.loc[pd.to_datetime(account.signal_date).between(plan['start'],plan['end'])].set_index('signal_date');daily=part.join(descriptors,validate='one_to_one').join(raw_cash,validate='one_to_one');assert len(daily)==64 and daily.positive_weight5.notna().all() and daily.frozen_current_cash_weight.notna().all()
 assert (pd.to_datetime(daily.index)<pd.to_datetime(daily.trade_date)).all();assert (pd.to_datetime(daily.filing_date)<pd.to_datetime(daily.index)).all()
 daily['month']=daily.index.to_period('M').astype(str);daily['breadth_state']=np.where(daily.positive_weight5>.5,'majority_weight_up','not_majority_weight_up');all_daily[str(cost)]=daily
 for month,sub in daily.groupby('month'):
  monthly.append({'slippage_bps':cost,'month':month,'sessions':len(sub),'mean_raw_cash_pct':100*sub.frozen_current_cash_weight.mean(),'mean_actual_cash_pct':100*sub.cash_weight_after_open.mean(),'mean_positive_weight5_pct':100*sub.positive_weight5.mean(),'majority_weight_up_days':int(sub.positive_weight5.gt(.5).sum()),'maximum_new_low_weight20_pct':100*sub.new_low_weight20.max(),'mean_new_low_weight20_pct':100*sub.new_low_weight20.mean(),'median_downside_top5_share_pct':100*sub.downside_top5_share.median(),**sub[components].sum().to_dict(),'relative_wealth_change_pct':100*np.expm1(sub.relative_log.sum())})
 for state,sub in daily.groupby('breadth_state'):
  groups.append({'slippage_bps':cost,'breadth_state':state,'sessions':len(sub),'mean_raw_cash_pct':100*sub.frozen_current_cash_weight.mean(),'mean_actual_cash_pct':100*sub.cash_weight_after_open.mean(),**sub[components].sum().to_dict()})
 np.testing.assert_allclose(pd.DataFrame([r for r in monthly if r['slippage_bps']==cost])[components].sum(),part[components].sum(),atol=1e-12)
summary={'experiment':card.name,'decision':'descriptive_Q3_failure_mechanisms_no_predictive_selection','new_fits':0,'new_backtests':0,'outcome_correlations':0,'threshold_searches':0,'objective_achieved':False,'sessions':64,'provider_uri':plan['price_provider'],'region':'us','as_of':plan['end'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Predetermined quarter, all months and both majority descriptions retained; stock descriptors known before following account decision; accounting attribution is not causal removable loss or predictive validation','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/descriptors']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='quarter_mechanism_diagnosis',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan)
 rec.save_objects(artifact_path='analysis',daily=all_daily,descriptors=descriptors,monthly=pd.DataFrame(monthly),breadth_groups=pd.DataFrame(groups),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(monthly).to_csv(card/'monthly_diagnosis.csv',index=False);pd.DataFrame(groups).to_csv(card/'breadth_state_accounting.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':plan['price_provider'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(pd.DataFrame(monthly).to_string(index=False));print(pd.DataFrame(groups).to_string(index=False))


In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.analysis.breadth_change_decomposition import decompose_breadth_changes
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/q3_failure_mechanisms_20261014';plan=json.loads((card/'plan.json').read_text());summary=json.loads((card/'summary.json').read_text());parent=root/plan['sources']['constituents']['card'];ps=json.loads((parent/'summary.json').read_text());qlib.init(provider_uri=summary['provider_uri'],region='us')
with R.uri_context(ps['recorder_uri']):
 rec=R.get_recorder(recorder_id=ps['recorder_id'],experiment_name=parent.name);prices=rec.load_object('analysis/prices')['$adj_close'].unstack('instrument');weights=rec.load_object('analysis/weights')
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_name=card.name);descriptors=rec.load_object('analysis/descriptors')
minimum=prices.rolling(20,min_periods=20).min();indicator=prices.le(minimum).astype(float).where(minimum.notna()).reindex(descriptors.index)
w=weights.pivot(index='datetime',columns='instrument',values='weight_pct').reindex(index=descriptors.index,columns=prices.columns).fillna(0.)
decomposition=decompose_breadth_changes(indicator,w);np.testing.assert_allclose(decomposition.current,descriptors.new_low_weight20,atol=1e-12)
features=pd.DataFrame({'new_low_level20':decomposition.current,'new_low_price_contraction5':-decomposition.price_component.rolling(5,min_periods=5).sum(),'downside_top5_share':descriptors.downside_top5_share},index=descriptors.index)
features['contraction_known']=features.new_low_price_contraction5.notna().astype(float);features['concentration_known']=features.downside_top5_share.notna().astype(float)
frame=pd.concat({'feature':features},axis=1);frame.index=pd.MultiIndex.from_arrays([features.index,['DISCLOSED_SEMIS_BASKET']*len(features)],names=['datetime','instrument']);ds=DatasetH(handler=DataHandlerLP.from_df(frame),segments={'all':(None,None)});observed=ds.prepare('all',col_set='feature').droplevel('instrument');observed.index.name=features.index.name;pd.testing.assert_frame_equal(observed,features)
contract={'status':'unvalidated_research_descriptors_not_trading_signals','motivated_by_reused_failure_analysis':True,'new_fits':0,'original_closed_breadth_route_reopened':False,'fields':{'new_low_level20':'Reported equity-weight share at current20-session closing low; damage occurrence level, not probability','new_low_price_contraction5':'Negative sum of previous five available daily price-indicator components at prior-known baskets; positive means fewer new-low indicators; rolling-window effects remain, not proof of lower future risk','downside_top5_share':'Top5 negative current-day weighted stock-return contributions divided by all negative contributions; unknown if none','contraction_known':'No zero fill for initial five unavailable changes','concentration_known':'No zero fill for days without negative contribution'},'windows':'20/5/top5 inherited from earlier fixed descriptor definitions; no horizon search','availability':'after signal-day close for next-session research decision; historical first receipt not certified'}
summary['candidate_feature_contract']='candidate_feature_contract.json';summary['candidate_feature_rows']=len(features);summary['candidate_known_counts']={c:int(features[c].notna().sum()) for c in ['new_low_level20','new_low_price_contraction5','downside_top5_share']}
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_name=card.name);rec.save_objects(artifact_path='analysis',candidate_features=features,candidate_contract=contract,summary=summary)
 artifacts=Path(summary['recorder_uri'])/summary['experiment_id']/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id']);(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'candidate_feature_contract.json').write_text(json.dumps(contract,indent=2)+'\n');print(summary['candidate_known_counts'])


# 2026Q3差区间：防守与修复的时间错位需要分阶段看

0拟合、0新增回测、0参数搜索。完整保留预先选定差季度64个交易日、三个自然月及2/5bp两条原账户；成员描述量以信号日收盘信息计算，严格早于对应持仓决策。持仓篮子按此前已披露报告使用，不是官方每日指数名单。

|2bp原账户|7月|8月|9月|
|---|---:|---:|---:|
|原模型现金均值%|24.35|29.36|68.41|
|实际现金均值%|57.50|73.02|77.19|
|近5日上涨成分权重均值%|30.23|53.56|62.29|
|20日新低权重均值%|24.42|6.26|9.21|
|相对SOXX财富变化%|-3.67|+5.42|-12.23|

相对财富变化是策略增长倍数除以SOXX增长倍数减1，不是收益百分点。7月一度91.01%的篮子权重触及20日新低，不能把该月弱势仅归咎少数股票；SOXL相对贡献-0.08073对数单位，现金两项+0.04388，防守在会计上抵消了部分损失。8月现金贡献+0.05288，不能据季度总体落后就认定风控始终无用。9月主要损失为现金贡献-0.13614，原模型现金本身已68.41%，不能全部归因后置控制器。

5bp月度相对财富变化约-3.70%、+5.40%、-12.27%，同样体现阶段差异。季度2bp相对对数贡献合计-0.11498，与原季度归因核对一致。下跌贡献前5名集中度中位数7/8/9月约59.07%/82.96%/80.72%；它是负贡献集合内部比例，受成分权重和下跌股票数量影响，不是总市场风险或未来损失概率。

防止事后简化：整个季度多数权重成分过去5日上涨的31日，2bp相对对数贡献为+0.09424；其余33日为-0.20921。因此月度现象不能推出“广度高就该减现金”的规则；历史状态与后续收益之间并非单调，且季度本身是按失败结果选来解释的。

保留三项未验证描述：新低发生率水平、去除篮子更新影响的五日新低指标收敛、负贡献前5名集中度。窗口沿用既有20/5/top5，不另选周期。五日价格指标收敛仍含滚动窗口效应，不能等同未来风险必降；首五个变化窗口未知，不填0；没有负贡献时集中度未知。研究表经Qlib Dataset读回，完整值及掩码仅存Recorder。

决定：descriptive_Q3_failure_mechanisms_no_predictive_selection。所需信息应区分“当前损伤程度”与“修复方向”，但本卡没有证明新增描述有增量预测能力，没有把它们接入模型，没有重开已失败的旧广度模型。下一步应在不以收益调参的前提下核查这些描述跨区间是否一致、是否只是旧输入的重复表达。稳定净超额仍未实现，生产release未改。

[全部月份](monthly_diagnosis.csv) · [完整状态分组](breadth_state_accounting.csv) · [未验证特征合同](candidate_feature_contract.json)

Recorder `9d412c8519ff41eaa140ae72b28321c2`。
